In [1]:
import torch
from pathlib import Path
from scipy.constants import pi
from ase.io import read
import numpy as np

from graph_longrange.kspace import compute_k_vectors_flat
from graph_longrange.energy import GTOElectrostaticEnergy
from graph_longrange.gto_utils import gto_basis_kspace_cutoff
from ase.build import bulk
from pymatgen.core import Structure
from pymatgen.analysis.energy_models import EwaldElectrostaticModel
from joblib import Memory
from mp_api.client import MPRester
from pymatgen.core.surface import SlabGenerator


torch.set_default_dtype(torch.float64)


/Users/tw/miniforge3/envs/surfpes/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _Jd, _W3j_flat, _W3j_indices = torch.load

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/Users/tw/miniforge3/envs/surfpes/lib/python3.11/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/Users/tw/miniforge3/envs/surfpes/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <EB3FF92A-5EB1-3EE8-AF8B-5923C1265422> /Users/tw/miniforge3/envs/surfpes/lib/python3.11/site-packages/torchvision/image.so
  Reason: tried: '/Users/tw/miniforge3/envs/surfpes/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/tw/miniforge3/envs/surfpes/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/tw/miniforge3/envs/surfpes/lib/python3.11/lib-dynload/../../libjpeg.9.dylib' (no such file), '/Users/tw/miniforge3/envs/surfpes/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wro

In [2]:
def atoms_list_to_batch(atoms_list):
    positions = []
    batch = []
    cells = []
    volumes = []
    pbcs = []

    for graph_i, atoms in enumerate(atoms_list):
        pos = torch.tensor(atoms.get_positions(), dtype=torch.get_default_dtype())
        positions.append(pos)
        batch.append(torch.full((pos.shape[0],), graph_i, dtype=torch.long))

        cell = torch.tensor(atoms.cell.array, dtype=torch.get_default_dtype())
        if torch.allclose(cell, torch.zeros_like(cell)):
            cell = torch.eye(3, dtype=cell.dtype)
        cells.append(cell)
        volumes.append(abs(torch.det(cell)))
        pbcs.append(torch.tensor(atoms.pbc, dtype=torch.bool))

    return (
        torch.cat(positions, dim=0),
        torch.cat(batch, dim=0),
        torch.stack(cells, dim=0),
        torch.stack(volumes, dim=0),
        torch.stack(pbcs, dim=0),
    )

In [3]:
# density_max_l is the multipole order. 0=charges, 1=charges+dipoles, etc...
# the realspace evalation is currently supported for only l<=1. 
density_max_l = 0

# this is sigma_n in the GTO basis, we generally use quite wide gaussians in ML models
density_smearing_width = 0.25

# use this function for a heuristic estimate of the k-space cutoff, 
# which is needed to determine the number of k-vectors to use in the reciprocal space sum.
KSPACE_CUTOFF = 3 * gto_basis_kspace_cutoff(
    sigmas=[density_smearing_width],
    max_l=density_max_l,
)

ENERGY_BLOCK = GTOElectrostaticEnergy(
    density_max_l=density_max_l,
    density_smearing_width=density_smearing_width,
    kspace_cutoff=KSPACE_CUTOFF,
    include_self_interaction=False,
)

In [4]:
def get_ewald_energy(structure: Structure, pbc: list[bool]):
    atoms = structure.to_ase_atoms()
    atoms.set_pbc(pbc)

    multipoles = atoms.arrays['oxi_states']
    multipoles = torch.tensor(multipoles).view(-1, 1)


    positions, batch, cell, volume, pbc = atoms_list_to_batch([atoms])

    r_cell = 2 * pi * torch.linalg.inv(cell).transpose(-1, -2)
    k_vectors, k_norm2, k_vector_batch, k0_mask = compute_k_vectors_flat(
        cutoff=KSPACE_CUTOFF,
        cell_vectors=cell,
        r_cell_vectors=r_cell,
)

    energy = ENERGY_BLOCK(
        k_vectors=k_vectors,
        k_norm2=k_norm2,
        k_vector_batch=k_vector_batch,
        k0_mask=k0_mask,
        source_feats=multipoles,
        node_positions=positions,
        batch=batch,
        volume=volume,
        pbc=pbc,
    )

    return energy

In [5]:
memory = Memory('.cachedir')

def view_structure(structure, viewer='ase'):
    return view(structure.to_ase_atoms(), viewer=viewer)

@memory.cache
def get_structure(mp_id):
    with MPRester() as mpr:
        # docs = mpr.materials.summary.search(material_ids=[mp_id], fields=["structure"])
        # structure = docs[0].structure
        # # -- Shortcut for a single Materials Project ID:
        structure = mpr.get_structure_by_material_id(mp_id,)
    
    return structure

structure = get_structure('mp-1265')
structure = structure.to_conventional()


slab_gen = SlabGenerator(structure, [1,1,1], 50, 200, primitive=True, max_normal_search=5) ### If you want the slab to be center set center_slab=True
slab = slab_gen.get_slab()
slab.add_oxidation_state_by_guess()
None

In [6]:
get_ewald_energy(slab, [True, True, True])

tensor([-2108.5311])

In [7]:
model = EwaldElectrostaticModel()
model.get_energy(slab)

np.float64(-2108.531118547808)

In [8]:
get_ewald_energy(slab, [True, True, False])

tensor([-1625.3043])

In [9]:
from ewald import get_ewald_energy

get_ewald_energy(slab, [True, True, False])

tensor([-1625.3043])